# CoordPredictor Training Pipeline

This notebook contains the complete training pipeline for the `CoordPredictor` model,
which predicts gaze coordinates from segmentation features.

## Requirements

```bash
pip install numpy torch matplotlib torchinfo
```

## Pipeline Overview

1. Import required libraries
2. Define network components (depthwise separable conv, double conv, deconv)
3. Define CoordPredictor model
4. Define angular error calculation function
5. Load training data from npz files
6. Train the model
7. Evaluate and save the model

## 1. Import Required Libraries

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
import matplotlib.pyplot as plt
from torchinfo import summary
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Define Network Components

In [2]:
class depthwise_separable_conv(nn.Module):
    """MobileNet V1-style depthwise separable convolution block"""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels,
            bias=False
        )
        self.pointwise = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = F.relu(self.bn1(self.depthwise(x)))
        x = F.relu(self.bn2(self.pointwise(x)))
        return x

class double_conv2d_bn(nn.Module):
    """Double depthwise separable convolution block"""
    def __init__(self, in_channels, out_channels, kernel_size=3, strides=1, padding=1):
        super().__init__()
        self.conv1 = depthwise_separable_conv(
            in_channels, out_channels,
            kernel_size=kernel_size,
            stride=strides,
            padding=padding
        )
        self.conv2 = depthwise_separable_conv(
            out_channels, out_channels,
            kernel_size=kernel_size,
            stride=strides,
            padding=padding
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        return x

class deconv2d_bn(nn.Module):
    """Deconvolution block"""
    def __init__(self, in_channels, out_channels, kernel_size=2, strides=2):
        super().__init__()
        self.conv1 = nn.ConvTranspose2d(
            in_channels, out_channels,
            kernel_size=kernel_size,
            stride=strides, bias=True
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        return F.relu(self.bn1(self.conv1(x)))

## 3. Define CoordPredictor Model

In [3]:
class CoordPredictor(nn.Module):
    """
    Coordinate predictor network that maps segmentation features to gaze angles.

    Input: Segmentation output features [B, in_channels, H, W]
    Output: Gaze coordinates [B, 2] -> (yaw_angle, pitch_angle)
    """

    def __init__(self, in_channels):
        super().__init__()

        self.layer1_conv = nn.Sequential(
            double_conv2d_bn(in_channels, 16),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )

        self.layer2_conv = nn.Sequential(
            double_conv2d_bn(16, 32),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        self.layer3_conv = nn.Sequential(
            double_conv2d_bn(32, 64),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.regressor = nn.Linear(64, 2)

    def forward(self, x):
        x = F.max_pool2d(self.layer1_conv(x), 2)
        x = F.max_pool2d(self.layer2_conv(x), 2)
        x = F.max_pool2d(self.layer3_conv(x), 2)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        out = self.regressor(x)
        return out

def count_parameters(model):
    """Count the number of trainable parameters."""
    total_params = sum(
        p.numel() for name, p in model.named_parameters()
        if not name.startswith("mycustomlayer")
    )
    return total_params

model = CoordPredictor(in_channels=4)
print("="*50)
print("Model Architecture:")
print("="*50)
print(model)
print(f"\nNumber of parameters: {count_parameters(model):,}")

Model Architecture:
CoordPredictor(
  (layer1_conv): Sequential(
    (0): double_conv2d_bn(
      (conv1): depthwise_separable_conv(
        (depthwise): Conv2d(4, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=4, bias=False)
        (pointwise): Conv2d(4, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (conv2): depthwise_separable_conv(
        (depthwise): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
        (pointwise): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (1): BatchNorm2d(16, eps=1e-05, momentu

## 4. Angular Error Calculation

In [4]:
def calculate_angular_error(predictions, labels):
    """
    Calculate angular error in degrees between predicted and true gaze vectors.

    Args:
        predictions: Tensor of shape [Batch_size, 2] with (yaw, pitch) in degrees
        labels: Tensor of shape [Batch_size, 2] with (yaw, pitch) in degrees

    Returns:
        angular_error_deg: Tensor of shape [Batch_size] with angular errors in degrees
    """
    pred_rad = predictions * (np.pi / 180.0)
    label_rad = labels * (np.pi / 180.0)

    yaw_pred, pitch_pred = pred_rad[:, 0], pred_rad[:, 1]
    yaw_label, pitch_label = label_rad[:, 0], label_rad[:, 1]

    v_pred_x = torch.cos(pitch_pred) * torch.sin(yaw_pred)
    v_pred_y = torch.sin(pitch_pred)
    v_pred_z = torch.cos(pitch_pred) * torch.cos(yaw_pred)

    v_label_x = torch.cos(pitch_label) * torch.sin(yaw_label)
    v_label_y = torch.sin(pitch_label)
    v_label_z = torch.cos(pitch_label) * torch.cos(yaw_label)

    v_pred = torch.stack([v_pred_x, v_pred_y, v_pred_z], dim=1)
    v_label = torch.stack([v_label_x, v_label_y, v_label_z], dim=1)

    dot_product = torch.sum(v_pred * v_label, dim=1)
    dot_product = torch.clamp(dot_product, -1.0 + 1e-7, 1.0 - 1e-7)

    angular_error_rad = torch.acos(dot_product)
    angular_error_deg = angular_error_rad * (180.0 / np.pi)

    return angular_error_deg

## 5. Load Training Data

In [5]:
def load_npz_data(train_file, test_file):
    """
    Load training and testing data from npz files.

    Args:
        train_file: Path to training npz file
        test_file: Path to testing npz file

    Returns:
        X_train_tensor, Y_train_tensor, X_test_tensor, Y_test_tensor
    """
    print(f"Loading training data from: {train_file}")
    print(f"Loading testing data from: {test_file}")

    train_bundle = np.load(train_file)
    test_bundle = np.load(test_file)

    X_train_np = train_bundle['features']
    Y_train_np = train_bundle['labels']
    X_test_np = test_bundle['features']
    Y_test_np = test_bundle['labels']

    print(f"✅ Training set loaded: features {X_train_np.shape}, labels {Y_train_np.shape}")
    print(f"✅ Testing set loaded: features {X_test_np.shape}, labels {Y_test_np.shape}")

    X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32)
    Y_train_tensor = torch.tensor(Y_train_np, dtype=torch.float32) / 30
    X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32)
    Y_test_tensor = torch.tensor(Y_test_np, dtype=torch.float32) / 30

    return X_train_tensor, Y_train_tensor, X_test_tensor, Y_test_tensor

train_file = "new_train_dataset_with_coords2.npz"
test_file = "new_test_dataset_with_coords2.npz"

if os.path.exists(train_file) and os.path.exists(test_file):
    X_train, Y_train, X_test, Y_test = load_npz_data(train_file, test_file)

    train_dataset = TensorDataset(X_train, Y_train)
    test_dataset = TensorDataset(X_test, Y_test)

    batch_size = 4
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"\nDataLoaders created:")
    print(f"  Training batches: {len(train_loader)}")
    print(f"  Testing batches: {len(test_loader)}")
else:
    print(f"❌ Data files not found!")
    print(f"  Please generate {train_file} and {test_file} first.")
    print(f"  See create_new_dataset function in segmentation training notebook.")

Loading training data from: new_train_dataset_with_coords2.npz
Loading testing data from: new_test_dataset_with_coords2.npz
✅ Training set loaded: features (994, 4, 300, 300), labels (994, 2)
✅ Testing set loaded: features (427, 4, 300, 300), labels (427, 2)

DataLoaders created:
  Training batches: 249
  Testing batches: 107


## 6. Train the Model

In [6]:
def train_coord_predictor(model, train_loader, test_loader, device, num_epochs=500, lr=0.001):
    """
    Train the CoordPredictor model.

    Args:
        model: CoordPredictor instance
        train_loader: Training DataLoader
        test_loader: Testing DataLoader
        device: Training device (cuda or cpu)
        num_epochs: Number of training epochs
        lr: Learning rate

    Returns:
        model: Trained model
        training_history: Dictionary with training loss and error history
    """
    model.to(device)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    training_history = {
        'train_loss': [],
        'train_error': [],
        'val_loss': [],
        'val_error': []
    }

    best_val_error = float('inf')

    print(f"\n🚀 Starting training for {num_epochs} epochs...")

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        total_batch_error = 0

        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()

            predictions = model(batch_x)
            predictions_deg = predictions * 30
            batch_y_deg = batch_y * 30

            loss = criterion(predictions_deg, batch_y_deg)
            batch_error = calculate_angular_error(predictions_deg, batch_y_deg)
            total_batch_error += torch.sum(batch_error).item()

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        avg_error = total_batch_error / len(train_loader)

        training_history['train_loss'].append(avg_loss)
        training_history['train_error'].append(avg_error)

        model.eval()
        val_loss = 0
        val_error = 0

        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                predictions = model(batch_x)
                predictions_deg = predictions * 30
                batch_y_deg = batch_y * 30
                loss = criterion(predictions_deg, batch_y_deg)
                error = calculate_angular_error(predictions_deg, batch_y_deg)
                val_loss += loss.item()
                val_error += torch.sum(error).item()

        avg_val_loss = val_loss / len(test_loader)
        avg_val_error = val_error / len(test_loader)

        training_history['val_loss'].append(avg_val_loss)
        training_history['val_error'].append(avg_val_error)

        if avg_val_error < best_val_error:
            best_val_error = avg_val_error
            torch.save(model.state_dict(), 'best_coord_predictor.pth')

        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}]")
            print(f"  Train Loss: {avg_loss:.4f}, Train Error: {avg_error:.4f} deg")
            print(f"  Val Loss: {avg_val_loss:.4f}, Val Error: {avg_val_error:.4f} deg")

    print("🎉 Training complete!")
    print(f"Best validation error: {best_val_error:.4f} deg")

    return model, training_history

if 'train_loader' in dir() and 'test_loader' in dir():
    trained_model, history = train_coord_predictor(
        model=model,
        train_loader=train_loader,
        test_loader=test_loader,
        device=device,
        num_epochs=500,
        lr=0.001
    )
else:
    print("⚠️ DataLoaders not available. Please run the data loading cell first.")


🚀 Starting training for 500 epochs...


KeyboardInterrupt: 

## 7. Visualize Training Results

In [ ]:
def plot_training_history(history):
    """
    Plot training and validation loss/error curves.

    Args:
        history: Dictionary with training history
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history['train_loss'], label='Training Loss')
    axes[0].plot(history['val_loss'], label='Validation Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (MSE)')
    axes[0].set_title('Training vs Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history['train_error'], label='Training Error')
    axes[1].plot(history['val_error'], label='Validation Error')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Angular Error (degrees)')
    axes[1].set_title('Training vs Validation Angular Error')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

if 'history' in dir():
    plot_training_history(history)

## 8. Final Evaluation

In [ ]:
def evaluate_model(model, test_loader, device):
    """
    Evaluate the trained model on test set.

    Args:
        model: Trained CoordPredictor
        test_loader: Testing DataLoader
        device: Evaluation device

    Returns:
        metrics: Dictionary with evaluation metrics
    """
    model.load_state_dict(torch.load('best_coord_predictor.pth'))
    model.to(device)
    model.eval()

    all_predictions = []
    all_labels = []
    all_errors = []

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            predictions = model(batch_x)
            predictions_deg = predictions * 30
            labels_deg = batch_y * 30
            errors = calculate_angular_error(predictions_deg, labels_deg)

            all_predictions.append(predictions_deg.cpu().numpy())
            all_labels.append(labels_deg.cpu().numpy())
            all_errors.append(errors.cpu().numpy())

    all_predictions = np.concatenate(all_predictions)
    all_labels = np.concatenate(all_labels)
    all_errors = np.concatenate(all_errors)

    metrics = {
        'mean_error': np.mean(all_errors),
        'median_error': np.median(all_errors),
        'std_error': np.std(all_errors),
        'min_error': np.min(all_errors),
        'max_error': np.max(all_errors),
        'rmse_x': np.sqrt(np.mean((all_predictions[:, 0] - all_labels[:, 0]) ** 2)),
        'rmse_y': np.sqrt(np.mean((all_predictions[:, 1] - all_labels[:, 1]) ** 2))
    }

    print("="*60)
    print("Final Evaluation Results")
    print("="*60)
    print(f"Mean Angular Error: {metrics['mean_error']:.4f} deg")
    print(f"Median Angular Error: {metrics['median_error']:.4f} deg")
    print(f"Std Angular Error: {metrics['std_error']:.4f} deg")
    print(f"Min/Max Error: {metrics['min_error']:.4f} / {metrics['max_error']:.4f} deg")
    print(f"RMSE X: {metrics['rmse_x']:.4f} deg")
    print(f"RMSE Y: {metrics['rmse_y']:.4f} deg")
    print("="*60)

    return metrics, all_predictions, all_labels, all_errors

if 'test_loader' in dir():
    metrics, preds, labels, errors = evaluate_model(trained_model, test_loader, device)

## 9. Scatter Plot of Predictions vs Ground Truth

In [ ]:
def plot_predictions_vs_labels(preds, labels):
    """
    Plot predictions vs ground truth scatter plot.

    Args:
        preds: Predicted angles [N, 2]
        labels: Ground truth angles [N, 2]
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].scatter(labels[:, 0], preds[:, 0], alpha=0.6, label='X (Yaw)')
    axes[0].scatter(labels[:, 1], preds[:, 1], alpha=0.6, label='Y (Pitch)')

    min_val = min(labels.min(), preds.min())
    max_val = max(labels.max(), preds.max())
    axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.7,
                 label='Perfect Prediction')

    axes[0].set_xlabel('Ground Truth (degrees)')
    axes[0].set_ylabel('Predicted (degrees)')
    axes[0].set_title('Predictions vs Ground Truth')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_aspect('equal', adjustable='box')

    errors_x = preds[:, 0] - labels[:, 0]
    errors_y = preds[:, 1] - labels[:, 1]

    axes[1].hist(errors_x, bins=20, alpha=0.7,
                 label=f'X (RMSE={np.sqrt(np.mean(errors_x**2)):.4f})')
    axes[1].hist(errors_y, bins=20, alpha=0.7,
                 label=f'Y (RMSE={np.sqrt(np.mean(errors_y**2)):.4f})')
    axes[1].axvline(x=0, color='r', linestyle='--', alpha=0.7)
    axes[1].set_xlabel('Prediction Error (degrees)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Prediction Error Distribution')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

if 'preds' in dir() and 'labels' in dir():
    plot_predictions_vs_labels(preds, labels)